# Sesión 1. Fundamentos del forecasting

**Curso: Pronósticos Macroeconómicos con Machine Learning**

**Autor**: [Renato Vassallo](https://renatovassallo.github.io)

> **La idea que guía la sesión:** antes de comparar algoritmos hay que decidir qué se pronostica, cómo se representa la serie y qué patrones puede aprender cada modelo. El diseño del ejercicio importa más que el algoritmo de moda.

## Al terminar esta sesión podrás

1. distinguir tendencia, estacionalidad, ciclo y ruido;
2. usar el ADF como diagnóstico, sin convertirlo en un veredicto automático;
3. explicar qué sesgo inductivo aportan OLS y Random Forest;
4. reconocer cuándo la extrapolación y la no linealidad cambian la elección del modelo.

## Ruta de la sesión

| bloque | pregunta |
|---|---|
| Anatomía de una serie | ¿Qué historias se mezclan en una observación? |
| Representación | ¿Conviene modelar el nivel o una transformación? |
| OLS frente a Random Forest | ¿Qué puede aprender y extrapolar cada modelo? |
| Tres experimentos | ¿Qué cambia con una recta, un umbral y una tendencia? |

**Pregunta guía:** ¿el algoritmo falla o le pedimos aprender una estructura que no puede representar?

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Permite abrir la notebook desde la raíz del repositorio o desde session1/.
candidatos = [Path.cwd(), Path.cwd() / "session1", *Path.cwd().parents]
SESSION_DIR = next(
    (
        ruta.resolve()
        for ruta in candidatos
        if (ruta / "utils.py").exists() and (ruta / "data" / "monthly.csv").exists()
    ),
    None,
)
if SESSION_DIR is None:
    raise FileNotFoundError("No se encontró session1/data/monthly.csv.")

if str(SESSION_DIR) not in sys.path:
    sys.path.insert(0, str(SESSION_DIR))

import utils as U

U.set_style()
print(f"Datos de la sesión: {U.DATA}")


# Fundamentos

## 1. Una serie de tiempo mezcla varias historias

Una representación didáctica de una serie mensual es

$$y_t = T_t + S_t + C_t + \varepsilon_t,$$

donde $T_t$ es la **tendencia**, $S_t$ la **estacionalidad**, $C_t$ un **componente cíclico** y $\varepsilon_t$ el **ruido**.

Esta descomposición no es una verdad única: tendencia y ciclo no se observan por separado y dependen del método empleado. Primero simularemos un ejemplo en el que sí conocemos cada pieza.

**Antes de ejecutar:** ¿qué componente crees que dominará la serie observada?

In [ ]:
rng_componentes = np.random.default_rng(101)
n_meses = 180
tiempo = np.arange(n_meses)

tendencia = 10 + 0.03 * tiempo
estacionalidad = 1.2 * np.sin(2 * np.pi * tiempo / 12)

ciclo = np.zeros(n_meses)
for i in range(1, n_meses):
    ciclo[i] = 0.8 * ciclo[i - 1] + rng_componentes.normal(0, 0.6)

ruido = rng_componentes.normal(0, 0.4, n_meses)
serie_observada = tendencia + estacionalidad + ciclo + ruido

componentes = [
    (serie_observada, "serie observada  $y_t$", U.INK),
    (tendencia, "tendencia  $T_t$", U.ACCENT),
    (estacionalidad, "estacionalidad  $S_t$", U.GOLD),
    (ciclo, "ciclo AR(1)  $C_t$", U.OLIVE),
    (ruido, "ruido  $\\varepsilon_t$", U.MUTED),
]

fig, axes = plt.subplots(5, 1, figsize=(9.5, 7.2), sharex=True)
for i, (ax, (serie, etiqueta, color)) in enumerate(zip(axes, componentes)):
    ax.plot(tiempo, serie, color=color, lw=1.3)
    ax.set_ylabel(etiqueta, fontsize=8.5, rotation=0, ha="right", va="center")
    if i >= 2:
        ax.axhline(0, color=U.BORDER, lw=0.7)

axes[-1].set_xlabel("mes")
fig.suptitle(
    "Una serie de tiempo combina componentes con comportamientos distintos",
    x=0.02,
    ha="left",
    fontweight="bold",
)
U.save_fig(fig, "A01_componentes_serie")
plt.show()

## 2. El caso real: el IPC peruano

Reconstruimos un índice de precios a partir de sus variaciones mensuales y lo desestacionalizamos con **X-13ARIMA-SEATS**, el programa del US Census Bureau que usan las agencias estadísticas (BCRP, INEI, BEA, Eurostat). Lo corremos con `MacroPy`, que trae el binario oficial empaquetado.

El procedimiento tiene dos pasos: un modelo regARIMA extiende la serie con pronósticos y limpia efectos determinísticos (outliers), y la cascada de filtros X-11 separa tendencia-ciclo, estacionalidad e irregular. Para el IPC el ajuste resulta multiplicativo (transformación log automática), así que reportamos el factor estacional y el irregular en porcentaje del nivel.

**Importante:** esta descomposición usa toda la muestra y es descriptiva. El borde final se revisa cuando llegan datos nuevos, igual que en las publicaciones oficiales; no es una transformación en tiempo real. Si tu plataforma no tiene binario empaquetado, `seasonal_adjust(indice_ipc, method="stl")` entrega un contraste 100% Python.

In [ ]:
from MacroPy import seasonal_adjust

inflacion = U.load_inflation()
inflacion_mensual = inflacion["ipc_mom"].dropna()
indice_ipc = 100 * (1 + inflacion_mensual / 100).cumprod()

ajuste_x13 = seasonal_adjust(indice_ipc, method="x13")

# Bajo la transformación log automática, el factor estacional y el
# irregular se leen como porcentajes del nivel del índice.
factor_estacional = 100 * (ajuste_x13.observed / ajuste_x13.seasadj - 1)
irregular_pct = 100 * (ajuste_x13.irregular - 1)

componentes_x13 = [
    (ajuste_x13.observed, "índice IPC", U.INK),
    (ajuste_x13.trend, "tendencia-ciclo", U.ACCENT),
    (factor_estacional, "factor estacional, %", U.GOLD),
    (irregular_pct, "irregular, %", U.MUTED),
]

fig, axes = plt.subplots(4, 1, figsize=(9.5, 6.6), sharex=True)
inicio_borde = indice_ipc.index[-6]
for i, (ax, (serie, etiqueta, color)) in enumerate(zip(axes, componentes_x13)):
    ax.plot(serie.index, serie, color=color, lw=1.15)
    ax.set_ylabel(etiqueta, fontsize=8.5, rotation=0, ha="right", va="center")
    ax.axvspan(inicio_borde, indice_ipc.index[-1], color=U.TINT, alpha=0.35)
    if i >= 2:
        ax.axhline(0, color=U.BORDER, lw=0.8)

fig.suptitle(
    "X13 del IPC peruano: la descomposición que publican las agencias",
    x=0.02,
    ha="left",
    fontweight="bold",
)
axes[-1].set_xlabel("zona sombreada: el borde se revisa con cada dato nuevo")
U.save_fig(fig, "A02_x13_ipc")
plt.show()

### Cómo leer la figura

- La tendencia-ciclo absorbe los movimientos de baja frecuencia. X13 no separa "la" tendencia económica de "el" ciclo: entrega una sola curva suave.
- El factor estacional del IPC es pequeño (entre -0.6% y +0.6%) pero sistemático, y X-11 le permite evolucionar lentamente en el tiempo.
- El irregular recoge lo que ni la tendencia ni la estacionalidad explican: shocks puntuales de precios.
- Los últimos meses se revisan cuando llegan datos nuevos: el mismo problema de borde que enfrentan las agencias.

La descomposición ayuda a pensar qué estructura queremos modelar, pero todavía no decide si la representación elegida es estable.

## 3. Estacionariedad y raíz unitaria

Una serie es débilmente estacionaria si su media, varianza y autocovarianzas no cambian con el tiempo. Un paseo aleatorio,

$$y_t = y_{t-1} + \varepsilon_t,$$

acumula shocks de forma permanente y puede producir regresiones espurias cuando se combina sin cuidado con otros regresores no estacionarios.

El ADF estima una ecuación del tipo

$$\Delta y_t = \alpha + \gamma y_{t-1} + \sum_{i=1}^{p}\phi_i\Delta y_{t-i} + \varepsilon_t$$

y contrasta $H_0: \gamma=0$. Rechazar $H_0$ aporta evidencia contra una raíz unitaria bajo esa especificación. No prueba por sí solo que toda la distribución sea estable ni descarta quiebres de régimen.

In [ ]:
from statsmodels.tsa.stattools import adfuller

series_adf = {
    "IPC en nivel": indice_ipc,
    "Inflación anual": inflacion["ipc_yoy"].dropna(),
    "Inflación mensual": inflacion_mensual,
}

fig, axes = plt.subplots(3, 1, figsize=(9.5, 6.2), sharex=True)
for ax, (nombre, serie), color in zip(axes, series_adf.items(), [U.INK, U.ACCENT, U.OLIVE]):
    ax.plot(serie.index, serie, color=color, lw=1.05)
    ax.set_ylabel(nombre, fontsize=8.5, rotation=0, ha="right", va="center")

axes[-1].set_xlabel("fecha")
fig.suptitle("La transformación cambia la geometría de la serie", x=0.02, ha="left")
U.save_fig(fig, "A03_transformaciones_estacionariedad")
plt.show()

resultados_adf = []
for nombre, serie in series_adf.items():
    estadistico, p_valor, rezagos, n_obs, *_ = adfuller(serie, autolag="AIC")
    resultados_adf.append(
        {
            "serie": nombre,
            "n": n_obs,
            "rezagos": rezagos,
            "ADF": estadistico,
            "p_valor": p_valor,
            "decisión al 5%": "rechaza H0" if p_valor < 0.05 else "no rechaza H0",
        }
    )

tabla_adf = pd.DataFrame(resultados_adf)
tabla_adf["ADF"] = tabla_adf["ADF"].map(lambda valor: f"{valor:.2f}")
tabla_adf["p_valor"] = tabla_adf["p_valor"].map(
    lambda valor: "< 0.001" if valor < 0.001 else f"{valor:.3f}"
)
print(tabla_adf.to_string(index=False))

> **Regla de trabajo del curso:** si el nivel muestra evidencia de integración o una tendencia difícil de extrapolar, modelaremos una transformación más estable o incorporaremos explícitamente esa estructura. El ADF es una pieza de evidencia, no un semáforo automático.

La variación anual elimina exactamente una estacionalidad determinística que se repite cada doce meses. Si el patrón estacional cambia, esa cancelación deja de ser exacta.

**Pregunta:** aunque el ADF rechace una raíz unitaria para la inflación anual, ¿usarías sin más los mismos parámetros antes y después de 2021?

## 4. Dos sesgos inductivos: OLS y Random Forest

### Regresión lineal

Con información disponible en $x_t$, un forecast directo puede escribirse como

$$y_{t+h}=\beta_0+\beta'x_t+\varepsilon_{t+h}.$$

OLS elige los coeficientes que minimizan la suma de errores cuadrados. Su fortaleza es imponer una estructura parsimoniosa que puede extrapolar. Su riesgo es que esa forma lineal sea incorrecta.

Bajo exogeneidad estricta, la autocorrelación de los errores no sesga automáticamente los coeficientes, pero invalida los errores estándar convencionales. Si los regresores son endógenos, por ejemplo por una dinámica mal especificada, corregir la matriz de varianzas no recupera consistencia.

### Random Forest

Un árbol divide el espacio de features y predice una media dentro de cada hoja. Un bosque promedia muchos árboles construidos con muestras bootstrap y, cuando hay varias features, subconjuntos aleatorios de variables.

- La predicción es **constante por tramos**.
- Más árboles estabilizan el promedio, pero no crean capacidad de extrapolación.
- La predicción es un promedio de valores observados del target y permanece dentro de su rango de entrenamiento.

| propiedad | OLS | Random Forest |
|---|---|---|
| Forma | lineal y global | flexible y local |
| Extrapolación | sí, si la estructura lineal es creíble | no |
| Umbrales e interacciones | requieren features explícitas | puede aprenderlos dentro del rango visto |
| Interpretación | coeficientes directos | curva de respuesta e importancias asociativas |

Un coeficiente es una asociación marginal en las unidades de la especificación. Solo es una elasticidad en casos como una regresión log-log, y no es causal sin una estrategia de identificación.

## 5. Experimento 1: una relación lineal

Generamos datos con

$$y=2+0.8x+\varepsilon, \qquad x\in[-3,3].$$

Entrenamos dentro de ese rango y pedimos predicciones entre $-6$ y $6$.

**Predicción antes de ejecutar:** ¿qué modelo tendrá menor error dentro del rango? ¿Qué ocurrirá fuera?

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

rng_lineal = np.random.default_rng(202)
n_train = 200
x_train = rng_lineal.uniform(-3, 3, n_train)
y_train = 2 + 0.8 * x_train + rng_lineal.normal(0, 0.5, n_train)
X_train = x_train.reshape(-1, 1)

modelo_ols = LinearRegression().fit(X_train, y_train)
modelo_rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=4,
    random_state=202,
).fit(X_train, y_train)

x_grid = np.linspace(-6, 6, 600)
X_grid = x_grid.reshape(-1, 1)
relacion_verdadera = 2 + 0.8 * x_grid

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.scatter(x_train, y_train, s=11, color=U.MUTED, alpha=0.35, label="datos de entrenamiento")
ax.plot(x_grid, relacion_verdadera, color=U.INK, lw=1.2, ls="dashed", label="relación verdadera")
ax.plot(x_grid, modelo_ols.predict(X_grid), color=U.ACCENT, lw=2, label="OLS")
ax.plot(x_grid, modelo_rf.predict(X_grid), color=U.OLIVE, lw=1.7, label="Random Forest")
ax.axvspan(-6, -3, color=U.TINT, alpha=0.5)
ax.axvspan(3, 6, color=U.TINT, alpha=0.5)
ax.axvline(-3, color=U.BORDER, lw=0.9)
ax.axvline(3, color=U.BORDER, lw=0.9)
ax.set_title("Dentro del rango el bosque interpola; fuera del rango se aplana", loc="left")
ax.set_xlabel("x, sombreado = extrapolación")
ax.set_ylabel("y")
ax.legend(fontsize=9)
U.save_fig(fig, "A04_ols_rf_extrapolacion")
plt.show()

dentro = np.abs(x_grid) <= 3
evaluacion = []
for nombre, modelo in [("OLS", modelo_ols), ("Random Forest", modelo_rf)]:
    prediccion = modelo.predict(X_grid)
    evaluacion.append(
        {
            "modelo": nombre,
            "RMSE dentro": U.rmse(prediccion[dentro] - relacion_verdadera[dentro]),
            "RMSE fuera": U.rmse(prediccion[~dentro] - relacion_verdadera[~dentro]),
        }
    )

tabla_lineal = pd.DataFrame(evaluacion).set_index("modelo").round(3)
print(tabla_lineal.to_string())
print(f"Pendiente OLS estimada: {modelo_ols.coef_[0]:.3f}; pendiente verdadera: 0.800")

### Lectura

El bosque aproxima la relación dentro del soporte observado. Fuera de ese soporte no prolonga la pendiente: reutiliza promedios de hojas extremas. Aumentar el número de árboles reduce la inestabilidad del promedio, pero no cambia esta limitación estructural.

## 6. Experimento 2: una relación con umbral

Ahora la pendiente cambia cuando $x$ supera 1:

$$y=0.5x+2.5\max(x-1,0)+\varepsilon.$$

Comparamos dos modelos:

1. OLS global, que impone una única pendiente;
2. Random Forest, que busca cortes sin recibir el umbral.

Este es un ejercicio de recuperación de la relación **dentro del rango observado**, no todavía un backtest temporal.

In [ ]:
rng_umbral = np.random.default_rng(303)
x_umbral_train = rng_umbral.uniform(-3, 3, 240)
y_umbral_train = (
    0.5 * x_umbral_train
    + 2.5 * np.maximum(x_umbral_train - 1, 0)
    + rng_umbral.normal(0, 0.4, len(x_umbral_train))
)

X_global = x_umbral_train.reshape(-1, 1)
ols_global = LinearRegression().fit(X_global, y_umbral_train)
rf_umbral = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=4,
    random_state=303,
).fit(X_global, y_umbral_train)

x_umbral_grid = np.linspace(-3, 3, 500)
X_umbral_grid = x_umbral_grid.reshape(-1, 1)
verdad_umbral = 0.5 * x_umbral_grid + 2.5 * np.maximum(x_umbral_grid - 1, 0)

predicciones_umbral = {
    "OLS global": ols_global.predict(X_umbral_grid),
    "Random Forest": rf_umbral.predict(X_umbral_grid),
}

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.scatter(x_umbral_train, y_umbral_train, s=11, color=U.MUTED, alpha=0.35, label="datos")
ax.plot(x_umbral_grid, verdad_umbral, color=U.INK, lw=1.2, ls="dashed", label="relación verdadera")
ax.plot(x_umbral_grid, predicciones_umbral["OLS global"], color=U.ACCENT, lw=1.8, label="OLS global")
ax.plot(x_umbral_grid, predicciones_umbral["Random Forest"], color=U.OLIVE, lw=1.6, label="Random Forest")
ax.axvline(1, color=U.BORDER, lw=1.1)
ax.text(1.04, ax.get_ylim()[0] + 0.2, "umbral verdadero", fontsize=8.5, color=U.MUTED)
ax.set_title("Con un umbral, una única recta se equivoca en ambos regímenes", loc="left")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(fontsize=8.5)
U.save_fig(fig, "A05_ols_rf_umbral")
plt.show()

tabla_umbral = pd.DataFrame(
    {
        nombre: {"RMSE vs relación verdadera": U.rmse(pred - verdad_umbral)}
        for nombre, pred in predicciones_umbral.items()
    }
).T.round(3)
print(tabla_umbral.to_string())

### Lectura

OLS global impone una única pendiente y promedia los dos regímenes. Random Forest aproxima el cambio de pendiente mediante cortes sucesivos, sin recibir la ubicación del umbral.

La ventaja del bosque depende de haber observado suficientes datos a ambos lados del quiebre. Un régimen completamente nuevo sigue estando fuera de su soporte de entrenamiento.

## 7. Experimento 3: tendencia y extrapolación temporal

Construimos una serie con tendencia determinística y ruido:

$$y_t=0.05t+\varepsilon_t.$$

Entrenamos hasta $t=140$ y, desde ese único origen, proyectamos toda la trayectoria $t=141,\ldots,179$. Los modelos reciben únicamente el tiempo, que sí se conoce en el origen. No se utilizan realizaciones futuras de $y$ como features.

Esta simplificación aísla la pregunta: ¿qué hace cada modelo cuando el test queda fuera del rango temporal observado?

In [ ]:
rng_tendencia = np.random.default_rng(404)
n_tiempo = 180
t_tendencia = np.arange(n_tiempo)
media_verdadera = 0.05 * t_tendencia
y_tendencia = media_verdadera + rng_tendencia.normal(0, 0.5, n_tiempo)

datos_tendencia = pd.DataFrame({"t": t_tendencia, "y": y_tendencia})
train_tendencia = datos_tendencia[datos_tendencia["t"] <= 140]
test_tendencia = datos_tendencia[datos_tendencia["t"] > 140]

ols_tendencia = LinearRegression().fit(train_tendencia[["t"]], train_tendencia["y"])
rf_tendencia = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=4,
    random_state=404,
).fit(train_tendencia[["t"]], train_tendencia["y"])

pred_ols_tendencia = ols_tendencia.predict(test_tendencia[["t"]])
pred_rf_tendencia = rf_tendencia.predict(test_tendencia[["t"]])

fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(datos_tendencia["t"], datos_tendencia["y"], color=U.MUTED, lw=1, label="serie observada")
ax.plot(t_tendencia, media_verdadera, color=U.INK, lw=1.1, ls="dashed", label="media verdadera")
ax.plot(test_tendencia["t"], pred_ols_tendencia, color=U.ACCENT, lw=2, label="forecast OLS")
ax.plot(test_tendencia["t"], pred_rf_tendencia, color=U.OLIVE, lw=2, label="forecast Random Forest")
ax.axvspan(140, 179, color=U.TINT, alpha=0.28)
ax.axvline(140, color=U.BORDER, lw=1.2)
ax.text(142, ax.get_ylim()[0] + 0.25, "forecast desde un origen", fontsize=8.5, color=U.MUTED)
ax.set_title("El bosque no prolonga una tendencia fuera del rango temporal", loc="left")
ax.set_xlabel("t, sombreado = test")
ax.set_ylabel("y")
ax.legend(fontsize=8.5)
U.save_fig(fig, "A06_tendencia_extrapolacion")
plt.show()

tabla_tendencia = pd.DataFrame(
    {
        "OLS": {
            "RMSE test": U.rmse(test_tendencia["y"] - pred_ols_tendencia),
            "máximo forecast": pred_ols_tendencia.max(),
        },
        "Random Forest": {
            "RMSE test": U.rmse(test_tendencia["y"] - pred_rf_tendencia),
            "máximo forecast": pred_rf_tendencia.max(),
        },
    }
).T.round(3)
print(tabla_tendencia.to_string())
print(f"Máximo y observado en train: {train_tendencia['y'].max():.3f}")

### Lectura

OLS prolonga la estructura lineal que estimó. El bosque repite la media de sus hojas extremas y se aplana. Esto no convierte a OLS en un extrapolador infalible: si la tendencia cambia, su prolongación también falla.

En problemas con rezagos y horizontes mayores que uno hay que elegir además entre una estrategia recursiva y una estrategia directa. Esa decisión requiere un protocolo temporal explícito y aparecerá más adelante en el curso.

## 8. Síntesis de la sesión

| situación | punto de partida razonable | cautela |
|---|---|---|
| Relación aproximadamente lineal, muestra pequeña | modelo lineal | extrapola solo si la estructura permanece estable |
| Umbrales o interacciones observados | Random Forest | un bosque no aprende un régimen nunca visto |
| Target con tendencia | transformar o modelar la tendencia explícitamente | ningún modelo extrapola bien un quiebre desconocido |
| Muchas features y pocas observaciones | modelo lineal regularizado | la penalización se elige dentro de validación temporal |
| Necesidad de explicación | modelo parsimonioso, curvas de respuesta | interpretación predictiva no equivale a causalidad |

> **Conclusión:** no existe un ganador independiente del problema. La representación del target, el soporte observado y el conjunto de información determinan qué sesgo inductivo resulta útil.

## Comprobación final

1. ¿Puede el Random Forest pronosticar un valor del target por encima del máximo observado en train?
2. ¿Qué permite concluir y qué no permite concluir el ADF?
3. ¿Por qué una única recta falla cuando la pendiente cambia entre regímenes?
4. ¿Qué información futura habría contaminado el tercer experimento si hubiéramos usado rezagos realizados del test?

# Cierre: lo que viene

En esta sesión decidimos qué representación y qué sesgo inductivo necesita cada problema. En la **Sesión 2** construiremos el pipeline completo con la inflación peruana: un conjunto de features disponible en cada fecha, train, validación y test sin viajar en el tiempo, y una carrera de modelos contra el mismo benchmark con la misma información.

Notebook: `session2/s2_pipeline.ipynb`